# Data Wrangling with direct LLM asked data
Starting point:
- around 1000 datasets (23 tasks per model (12 tasks plus the flipped version, except for care)) with logits and probs for all answer alternatives of each subtask and the generated answers. 

What does this script do
- Read all survey data sets (flipped, and not flipped)
- use argmax to extract max logit and max probability for each item
- make two version:
    - one version where reverse reversed items, one without
    - flipped answers separately, flip back plus do not flip back
    - add subcategories for scales that have them
- save all df versions for then analysing them

Specials:
- for every scale I compared the frey materials quest_raw.csv and quest_proc.csv and tried to trace back how they got from one to the other. Then I did the same transformations
- therefore, for some tasks the scores are mapped onto a different scale (point system depending on given answer) on the `human_number` level (before mapped to LLM assigned probabilities)

Goal:

- have one value per item per model in different version (flipped, reversed, normal, with context, no context, logit argmax, generated answer)
- next script: then transform those values in "outcomes" for each subscale (like Frey did) and then have 36 values per model! (one per (sub-) scale).

## Packages & Helpers

In [ ]:
# packages
import pandas as pd
import numpy as np
import glob
import os
import matplotlib.pyplot as plt
from utils import load_dataframes, add_argmax_column

PATH = "../../data/raw/risk_data/LLM_data/prompts_direct/"

## BARRATT SCALE

In [ ]:
# load data
BARRATT_data = load_dataframes(task_name="barratt", path = PATH)
BARRATT_data["experiment"] = "BARRATT"

In [ ]:
# separate test items (that were just in there for a basic concept test, do not belong to BARRATT data)
test_data = BARRATT_data[BARRATT_data["item_id"] > 30]
BARRATT_data = BARRATT_data[BARRATT_data["item_id"] <= 30]


In [ ]:
test_data["item_text"].unique()

In [ ]:
# Adding task specific categories to save in all data

# add item categories
item_to_category = {
    1: "BISn", 2: "BISm", 3: "BISm",  4: "BISm", 5: "BISa",  6: "BISa",  7: "BISn",  8: "BISn",  9: "BISa",  10: "BISn",
    11: "BISa", 12: "BISn", 13: "BISn",  14: "BISn", 15: "BISn",  16: "BISm",  17: "BISm",  18: "BISn",  19: "BISm",  20: "BISa",
    21: "BISm", 22: "BISm", 23: "BISm",  24: "BISa", 25: "BISm",  26: "BISa",  27: "BISn",  28: "BISa",  29: "BISn",  30: "BISm"
}

BARRATT_data["category"] = BARRATT_data["item_id"].map(item_to_category)

In [ ]:
# add argmax logit and probability coloumn
BARRATT_data = add_argmax_column(BARRATT_data)

In [ ]:

# add whether item was reverse coded
reverse_coded = {
    1: True, 2: False, 3: False,  4: False, 5: False,  6: False,  7: True,  8: True,  9: True,  10: True,
    11: False, 12: True, 13: True,  14: False, 15: True,  16: False,  17: False,  18: False,  19: False,  20: True,
    21: False, 22: False, 23: False,  24: False, 25: False,  26: False,  27: False,  28: False,  29: True,  30: True
    }

# ------ for normal data ------------
# Apply mapping row-wise based on item number
BARRATT_data["reverse_coded"] = BARRATT_data["item_id"].map(reverse_coded)



In [ ]:
# seperate the datasets
raw_barrat = BARRATT_data[BARRATT_data["flipped"] == False]
raw_barrat_flipped = BARRATT_data[BARRATT_data["flipped"] == True]
barrat_flipped_back = raw_barrat_flipped.copy()
barrat_flipped_back["model_answer"] = 5 - barrat_flipped_back["model_answer"]
barrat_flipped_back["logit_score"] = 5 - barrat_flipped_back["logit_score"]
barrat_flipped_back["prob_score"] = 5 - barrat_flipped_back["prob_score"]

In [ ]:
# reverse LLM answers (again) where the items where reversed phrased
barrat_reversed = raw_barrat.copy()

# flip back answers that where reverse coded
mask = (barrat_reversed["reverse_coded"] == True)
barrat_reversed.loc[mask, "model_answer"] = 5 - barrat_reversed.loc[mask, "model_answer"]
barrat_reversed.loc[mask, "logit_score"] = 5 - barrat_reversed.loc[mask, "logit_score"]
barrat_reversed.loc[mask, "prob_score"] = 5 - barrat_reversed.loc[mask, "prob_score"]
# drop reverse-coded column (not needed in final data)
#barrat_reversed = barrat_reversed.drop(columns=["reverse_coded"])


# ------ for re-flipped data ------------
# Apply mapping row-wise based on item number
barrat_flipped_back_reversed_back = barrat_flipped_back.copy()

# flip back answers that where reverse coded
mask = (barrat_flipped_back_reversed_back["reverse_coded"] == True)
barrat_flipped_back_reversed_back.loc[mask, "model_answer"] = 5 - barrat_flipped_back_reversed_back.loc[mask, "model_answer"]
barrat_flipped_back_reversed_back.loc[mask, "logit_score"] = 5 - barrat_flipped_back_reversed_back.loc[mask, "logit_score"]
barrat_flipped_back_reversed_back.loc[mask, "prob_score"] = 5 - barrat_flipped_back_reversed_back.loc[mask, "prob_score"]
# drop reverse-coded column (not needed in final data)
#barrat_flipped_back_reversed_back = barrat_flipped_back_reversed_back.drop(columns=["reverse_coded"])


In [ ]:
# organize outcoming dfs

# all reflipped and re-reversed together
all_data_reflipped_and_rereversed = pd.concat([barrat_reversed, barrat_flipped_back_reversed_back], ignore_index=True)
# all non-reflipped and non-re-reversed together
all_data_raw = BARRATT_data.copy()

# only non-flipped data (once re-reversed, once raw) 
no_flip_data_rereversed = barrat_reversed
no_flip_data_raw = raw_barrat

# only flipped data (once re-reversed, once raw) 
flip_data_rereversed = barrat_flipped_back_reversed_back
flip_data_raw = raw_barrat_flipped

## CARE TASK

In [ ]:
# load data
CARE_data = load_dataframes(task_name="care", path = PATH)
CARE_data["experiment"] = "CARE"

In [ ]:
# Adding task specific categories to save in all data
# add item categories
item_to_category = {
    1: "CAREa", 2: "CAREa", 3: "CAREa",  4: "CAREa", 5: "CAREa",  6: "CAREa",  7: "CAREa",  8: "CAREa",  9: "CAREa",  10: "CAREs",
    11: "CAREs", 12: "CAREs", 13: "CAREs",  14: "CAREs", 15: "CAREs",  16: "CAREw",  17: "CAREw",  18: "CAREw",  19: "CAREw"
}

CARE_data["category"] = CARE_data["item_id"].map(item_to_category)


In [ ]:
# add argmax logit and probability coloumn
CARE_data = add_argmax_column(CARE_data)

In [ ]:
CARE_data["flipped"].unique()

In [ ]:
# organize outcoming dfs

# all reflipped and re-reversed together
all_data_reflipped_and_rereversed = pd.concat([all_data_reflipped_and_rereversed, CARE_data], ignore_index=True)
# all non-reflipped and non-re-reversed together
all_data_raw = pd.concat([all_data_raw, CARE_data], ignore_index=True)

# only non-flipped data (once re-reversed, once raw) 
no_flip_data_rereversed = pd.concat([no_flip_data_rereversed, CARE_data], ignore_index=True)
no_flip_data_raw = pd.concat([no_flip_data_raw, CARE_data], ignore_index=True)

# only flipped data (once re-reversed, once raw) 
# ignore here?!?!?!?

## DAST SCALE

In [ ]:
# load data
DAST_data = load_dataframes(task_name="dast", path = PATH)
DAST_data["experiment"] = "DAST"

In [ ]:
# add argmax logit and probability coloumn
DAST_data = add_argmax_column(DAST_data)

In [ ]:
# seperate the datasets
raw_dast = DAST_data[DAST_data["flipped"] == False]
raw_dast_flipped = DAST_data[DAST_data["flipped"] == True]
dast_flipped_back = raw_dast_flipped.copy()
dast_flipped_back["model_answer"] = 3 - dast_flipped_back["model_answer"]
dast_flipped_back["logit_score"] = 3 - dast_flipped_back["logit_score"]
dast_flipped_back["prob_score"] = 3 - dast_flipped_back["prob_score"]

In [ ]:
# Define mappings for each DAST question:
dast_maps = {
    1: {1: 1, 2: 0},                           
    2: {1: 1, 2: 0},
    3: {1: 1, 2: 0},
    4: {1: 0, 2: 1},
    5: {1: 0, 2: 1},
    6: {1: 1, 2: 0},
    7: {1: 1, 2: 0},
    8: {1: 1, 2: 0},
    9: {1: 1, 2: 0},
    10: {1: 1, 2: 0},
    11: {1: 1, 2: 0},
    12: {1: 1, 2: 0},
    13: {1: 1, 2: 0},
    14: {1: 1, 2: 0},
    15: {1: 1, 2: 0},
    16: {1: 1, 2: 0},
    17: {1: 1, 2: 0},
    18: {1: 1, 2: 0},
    19: {1: 1, 2: 0},
    20: {1: 1, 2: 0}
}

# Apply mapping row-wise based on item number
def recode_dast_value(item_id, value):
    mapping = dast_maps.get(item_id)
    if mapping is not None:
        return mapping.get(value, None)
    return value

dast_reversed = raw_dast.copy()
dast_reversed["model_answer"] = dast_reversed.apply(lambda r: recode_dast_value(r["item_id"], r["model_answer"]), axis=1)
dast_reversed["logit_score"]  = dast_reversed.apply(lambda r: recode_dast_value(r["item_id"], r["logit_score"]), axis=1)
dast_reversed["prob_score"]   = dast_reversed.apply(lambda r: recode_dast_value(r["item_id"], r["prob_score"]), axis=1)

dast_flipped_back_reversed_back = dast_flipped_back.copy()
dast_flipped_back_reversed_back["model_answer"] = dast_flipped_back_reversed_back.apply(lambda r: recode_dast_value(r["item_id"], r["model_answer"]), axis=1)
dast_flipped_back_reversed_back["logit_score"]  = dast_flipped_back_reversed_back.apply(lambda r: recode_dast_value(r["item_id"], r["logit_score"]), axis=1)
dast_flipped_back_reversed_back["prob_score"]   = dast_flipped_back_reversed_back.apply(lambda r: recode_dast_value(r["item_id"], r["prob_score"]), axis=1)


In [ ]:
a = raw_dast[raw_dast["logit_score"].isna()]

In [ ]:
a

In [ ]:
dast_reversed["logit_score"].unique()

In [ ]:
# organize outcoming dfs

# all reflipped and re-reversed together
DAST_all_data_reflipped_and_rereversed = pd.concat([dast_reversed, dast_flipped_back_reversed_back], ignore_index=True)
all_data_reflipped_and_rereversed = pd.concat([all_data_reflipped_and_rereversed, DAST_all_data_reflipped_and_rereversed], ignore_index=True)
# all non-reflipped and non-re-reversed together
all_data_raw = pd.concat([all_data_raw, DAST_data], ignore_index=True)

# only non-flipped data (once re-reversed, once raw) 
no_flip_data_rereversed = pd.concat([no_flip_data_rereversed, dast_reversed], ignore_index=True)
no_flip_data_raw = pd.concat([no_flip_data_raw, raw_dast], ignore_index=True)

# only flipped data (once re-reversed, once raw) 
flip_data_rereversed = pd.concat([flip_data_rereversed, dast_flipped_back_reversed_back], ignore_index=True)
flip_data_raw = pd.concat([flip_data_raw, raw_dast_flipped], ignore_index=True)

## DM SCALE

In [ ]:
# load data
DM_data = load_dataframes(task_name="dm", path = PATH)
DM_data["experiment"] = "DM"

In [ ]:
# add argmax logit and probability coloumn
DM_data = add_argmax_column(DM_data)

In [ ]:
# seperate the datasets
raw_dm = DM_data[DM_data["flipped"] == False]
raw_dm_flipped = DM_data[DM_data["flipped"] == True]
dm_flipped_back = raw_dm_flipped.copy()
dm_flipped_back["model_answer"] = 5 - dm_flipped_back["model_answer"]
dm_flipped_back["logit_score"] = 5 - dm_flipped_back["logit_score"]
dm_flipped_back["prob_score"] = 5 - dm_flipped_back["prob_score"]

In [ ]:
# Define mappings for DM, so that all 4s are transformed to 1s (like in original Fey dataset):
# hier Abweichung von sonst Orientierung an Frey quest_proc df, aber da sonst später Umwandlung, hier gleich zu Scale 0-2
mapping = {
    4: 1,
    3: 2,
    2: 1,
    1: 0
} # here, I already go one step further than human process, directly to real scores that are then summed/averaged for final sum



dm_reversed = raw_dm.copy()
dm_reversed["model_answer"] = dm_reversed["model_answer"].map(mapping)
dm_reversed["logit_score"] = dm_reversed["logit_score"].map(mapping)
dm_reversed["prob_score"] = dm_reversed["prob_score"].map(mapping)



dm_flipped_back_reversed_back = dm_flipped_back.copy()
dm_flipped_back_reversed_back["model_answer"] = dm_flipped_back_reversed_back["model_answer"].map(mapping)
dm_flipped_back_reversed_back["logit_score"] = dm_flipped_back_reversed_back["logit_score"].map(mapping)
dm_flipped_back_reversed_back["prob_score"] = dm_flipped_back_reversed_back["prob_score"].map(mapping)


In [ ]:
# organize outcoming dfs

# all reflipped and re-reversed together
DM_all_data_reflipped_and_rereversed = pd.concat([dm_reversed, dm_flipped_back_reversed_back], ignore_index=True)
all_data_reflipped_and_rereversed = pd.concat([all_data_reflipped_and_rereversed, DM_all_data_reflipped_and_rereversed], ignore_index=True)
# all non-reflipped and non-re-reversed together
all_data_raw = pd.concat([all_data_raw, DM_data], ignore_index=True)

# only non-flipped data (once re-reversed, once raw) 
no_flip_data_rereversed = pd.concat([no_flip_data_rereversed, dm_reversed], ignore_index=True)
no_flip_data_raw = pd.concat([no_flip_data_raw, raw_dm], ignore_index=True)

# only flipped data (once re-reversed, once raw) 
flip_data_rereversed = pd.concat([flip_data_rereversed, dm_flipped_back_reversed_back], ignore_index=True)
flip_data_raw = pd.concat([flip_data_raw, raw_dm_flipped], ignore_index=True)

## DOSPERT SCALE

In [ ]:
# load data
DOSPERT_data = load_dataframes(task_name="dospert", path = PATH)
DOSPERT_data["experiment"] = "DOSPERT"

In [ ]:
# Adding task specific categories to save in all data

# add item categories
item_to_category = {
    1: "Social", 10: "Social", 16: "Social", 19: "Social", 23: "Social", 26: "Social", 34: "Social", 35: "Social",
    2: "Recreational", 6: "Recreational", 15: "Recreational", 17: "Recreational", 21: "Recreational", 31: "Recreational", 37: "Recreational", 38: "Recreational",
    3: "Gambling", 11: "Gambling", 22: "Gambling", 33: "Gambling",
    4: "Health", 8: "Health", 27: "Health", 29: "Health", 32: "Health", 36: "Health", 39: "Health", 40: "Health",
    5: "Ethical", 9: "Ethical", 12: "Ethical", 13: "Ethical", 14: "Ethical", 20: "Ethical", 25: "Ethical", 28: "Ethical",
    7: "Investment", 18: "Investment", 24: "Investment", 30: "Investment"
}

DOSPERT_data["category"] = DOSPERT_data["item_id"].map(item_to_category)


In [ ]:
# add argmax logit and probability coloumn
DOSPERT_data = add_argmax_column(DOSPERT_data)

In [ ]:
# seperate the datasets
raw_dospert = DOSPERT_data[DOSPERT_data["flipped"] == False]
raw_dospert_flipped = DOSPERT_data[DOSPERT_data["flipped"] == True]
dospert_flipped_back = raw_dospert_flipped.copy()
dospert_flipped_back["model_answer"] = 6 - dospert_flipped_back["model_answer"]
dospert_flipped_back["logit_score"] = 6 - dospert_flipped_back["logit_score"]
dospert_flipped_back["prob_score"] = 6 - dospert_flipped_back["prob_score"]

In [ ]:
# organize outcoming dfs

# all reflipped and re-reversed together
DOSPERT_all_data_reflipped = pd.concat([raw_dospert, dospert_flipped_back], ignore_index=True)
all_data_reflipped_and_rereversed = pd.concat([all_data_reflipped_and_rereversed, DOSPERT_all_data_reflipped], ignore_index=True)
# all non-reflipped and non-re-reversed together
all_data_raw = pd.concat([all_data_raw, DOSPERT_data], ignore_index=True)

# only non-flipped data (once re-reversed, once raw) 
no_flip_data_rereversed = pd.concat([no_flip_data_rereversed, raw_dospert], ignore_index=True)
no_flip_data_raw = pd.concat([no_flip_data_raw, raw_dospert], ignore_index=True)

# only flipped data (once re-reversed, once raw) 
flip_data_rereversed = pd.concat([flip_data_rereversed, dospert_flipped_back], ignore_index=True)
flip_data_raw = pd.concat([flip_data_raw, raw_dospert_flipped], ignore_index=True)

## GABS SCALE

In [ ]:
# load data
GABS_data = load_dataframes(task_name="gabs", path = PATH)
GABS_data["experiment"] = "GABS"

In [ ]:
# add argmax logit and probability coloumn
GABS_data = add_argmax_column(GABS_data)

In [ ]:
# seperate the datasets
raw_gabs = GABS_data[GABS_data["flipped"] == False]
raw_gabs_flipped = GABS_data[GABS_data["flipped"] == True]
gabs_flipped_back = raw_gabs_flipped.copy()
mask = (gabs_flipped_back["item_id"] == 0)
gabs_flipped_back.loc[mask, "model_answer"] = 3 - gabs_flipped_back.loc[mask, "model_answer"]
gabs_flipped_back.loc[mask, "logit_score"] = 3 - gabs_flipped_back.loc[mask, "logit_score"]
gabs_flipped_back.loc[mask, "prob_score"] = 3 - gabs_flipped_back.loc[mask, "prob_score"]

mask = (gabs_flipped_back["item_id"].isin(range(2,16)))
gabs_flipped_back.loc[mask, "model_answer"] = 5 - gabs_flipped_back.loc[mask, "model_answer"]
gabs_flipped_back.loc[mask, "logit_score"] = 5 - gabs_flipped_back.loc[mask, "logit_score"]
gabs_flipped_back.loc[mask, "prob_score"] = 5 - gabs_flipped_back.loc[mask, "prob_score"]


In [ ]:
# organize outcoming dfs

# all reflipped and re-reversed together
GABS_all_data_reflipped = pd.concat([raw_gabs, gabs_flipped_back], ignore_index=True)
all_data_reflipped_and_rereversed = pd.concat([all_data_reflipped_and_rereversed, GABS_all_data_reflipped], ignore_index=True)
# all non-reflipped and non-re-reversed together
all_data_raw = pd.concat([all_data_raw, GABS_data], ignore_index=True)

# only non-flipped data (once re-reversed, once raw) 
no_flip_data_rereversed = pd.concat([no_flip_data_rereversed, raw_gabs], ignore_index=True)
no_flip_data_raw = pd.concat([no_flip_data_raw, raw_gabs], ignore_index=True)

# only flipped data (once re-reversed, once raw) 
flip_data_rereversed = pd.concat([flip_data_rereversed, gabs_flipped_back], ignore_index=True)
flip_data_raw = pd.concat([flip_data_raw, raw_gabs_flipped], ignore_index=True)

## PRI SCALE

In [ ]:
# load data
PRI_data = load_dataframes(task_name="pri", path = PATH)
PRI_data["experiment"] = "PRI"

In [ ]:
# add argmax logit and probability coloumn
PRI_data = add_argmax_column(PRI_data)

In [ ]:
# seperate the datasets
raw_pri = PRI_data[PRI_data["flipped"] == False]
raw_pri_flipped = PRI_data[PRI_data["flipped"] == True]
pri_flipped_back = raw_pri_flipped.copy()
pri_flipped_back["model_answer"] = 3 - pri_flipped_back["model_answer"]
pri_flipped_back["logit_score"] = 3 - pri_flipped_back["logit_score"]
pri_flipped_back["prob_score"] = 3 - pri_flipped_back["prob_score"]

In [ ]:
# organize outcoming dfs

# all reflipped and re-reversed together
PRI_all_data_reflipped = pd.concat([raw_pri, pri_flipped_back], ignore_index=True)
all_data_reflipped_and_rereversed = pd.concat([all_data_reflipped_and_rereversed, PRI_all_data_reflipped], ignore_index=True)
# all non-reflipped and non-re-reversed together
all_data_raw = pd.concat([all_data_raw, PRI_data], ignore_index=True)

# only non-flipped data (once re-reversed, once raw) 
no_flip_data_rereversed = pd.concat([no_flip_data_rereversed, raw_pri], ignore_index=True)
no_flip_data_raw = pd.concat([no_flip_data_raw, raw_pri], ignore_index=True)

# only flipped data (once re-reversed, once raw) 
flip_data_rereversed = pd.concat([flip_data_rereversed, pri_flipped_back], ignore_index=True)
flip_data_raw = pd.concat([flip_data_raw, raw_pri_flipped], ignore_index=True)

## SOEP SCALE

In [ ]:
# load data
SOEP_data = load_dataframes(task_name="soep", path = PATH)
SOEP_data["experiment"] = "SOEP"

In [ ]:
# Adding task specific categories to save in all data

# add item categories
item_to_category = {
     1: "SOEP", 2: "SOEPdri", 3: "SOEPfin",  4: "SOEPrec", 5: "SOEPocc",  6: "SOEPhea",  7: "SOEPsoc"
}

SOEP_data["category"] = SOEP_data["item_id"].map(item_to_category)


In [ ]:
# add argmax logit and probability coloumn
SOEP_data = add_argmax_column(SOEP_data)

In [ ]:
# seperate the datasets
raw_soep = SOEP_data[SOEP_data["flipped"] == False]
raw_soep_flipped = SOEP_data[SOEP_data["flipped"] == True]
soep_flipped_back = raw_soep_flipped.copy()
soep_flipped_back["model_answer"] = 12 - soep_flipped_back["model_answer"]
soep_flipped_back["logit_score"] = 12 - soep_flipped_back["logit_score"]
soep_flipped_back["prob_score"] = 12 - soep_flipped_back["prob_score"]

In [ ]:
# organize outcoming dfs

# all reflipped and re-reversed together
SOEP_all_data_reflipped = pd.concat([raw_soep, soep_flipped_back], ignore_index=True)
all_data_reflipped_and_rereversed = pd.concat([all_data_reflipped_and_rereversed, SOEP_all_data_reflipped], ignore_index=True)
# all non-reflipped and non-re-reversed together
all_data_raw = pd.concat([all_data_raw, SOEP_data], ignore_index=True)

# only non-flipped data (once re-reversed, once raw) 
no_flip_data_rereversed = pd.concat([no_flip_data_rereversed, raw_soep], ignore_index=True)
no_flip_data_raw = pd.concat([no_flip_data_raw, raw_soep], ignore_index=True)

# only flipped data (once re-reversed, once raw) 
flip_data_rereversed = pd.concat([flip_data_rereversed, soep_flipped_back], ignore_index=True)
flip_data_raw = pd.concat([flip_data_raw, raw_soep_flipped], ignore_index=True)

## SSSV SCALE

In [ ]:
# load data
SSSV_data = load_dataframes(task_name="sssv", path = PATH)
SSSV_data["experiment"] = "SSSV"

In [ ]:
# add item categories
item_to_category = {
     3: "SStas", 11: "SStas", 16: "SStas", 17: "SStas", 20: "SStas", 21: "SStas", 23: "SStas", 28: "SStas", 38: "SStas", 40: "SStas",
     4: "SSexp", 6: "SSexp", 9: "SSexp", 10: "SSexp", 14: "SSexp", 18: "SSexp", 19: "SSexp", 22: "SSexp", 26: "SSexp", 37: "SSexp",
     1: "SSdis", 12: "SSdis", 13: "SSdis", 25: "SSdis", 29: "SSdis", 30: "SSdis", 32: "SSdis", 33: "SSdis", 35: "SSdis", 36: "SSdis",
     2: "SSbor", 5: "SSbor", 7: "SSbor", 8: "SSbor", 15: "SSbor", 24: "SSbor", 27: "SSbor", 31: "SSbor", 34: "SSbor", 39: "SSbor"
}

SSSV_data["category"] = SSSV_data["item_id"].map(item_to_category)

In [ ]:
# add argmax logit and probability coloumn
SSSV_data = add_argmax_column(SSSV_data)

In [ ]:
# add whether item was reverse coded
reverse_coded = {
     1: True, 2: False, 3: True, 4: False, 5: True, 6: True, 7: False, 8: True, 9: True, 10: False, 
     11: False, 12: False, 13: False, 14: True, 15: False, 16: True, 17: True, 18: True, 19: False, 20: False,
     21: False, 22: True, 23: True, 24: True, 25: False, 26: False, 27: False, 28: True, 29: True, 30: False,
     31: False, 32: True, 33: False, 34: True, 35: False, 36: True, 37: False, 38: False, 39: True, 40: False

}
SSSV_data["reverse_coded"] = SSSV_data["item_id"].map(reverse_coded)


In [ ]:
# seperate the datasets
raw_sssv = SSSV_data[SSSV_data["flipped"] == False]
raw_sssv_flipped = SSSV_data[SSSV_data["flipped"] == True]
sssv_flipped_back = raw_sssv_flipped.copy()
sssv_flipped_back["model_answer"] = 3 - sssv_flipped_back["model_answer"]
sssv_flipped_back["logit_score"] = 3 - sssv_flipped_back["logit_score"]
sssv_flipped_back["prob_score"] = 3 - sssv_flipped_back["prob_score"]

In [ ]:
# reverse human answers (again) where the items where reversed phrased

# ----- for normal data ----------
sssv_reversed = raw_sssv.copy()
# flip back answers that where reverse coded
mask = (sssv_reversed["reverse_coded"] == True)
sssv_reversed.loc[mask, "model_answer"] = 3 - sssv_reversed.loc[mask, "model_answer"]
sssv_reversed.loc[mask, "logit_score"] = 3 - sssv_reversed.loc[mask, "logit_score"]
sssv_reversed.loc[mask, "prob_score"] = 3 - sssv_reversed.loc[mask, "prob_score"]
# drop reverse-coded column (not needed in final data)
#sssv_reversed = sssv_reversed.drop(columns=["reverse_coded"])


# ----- for re-flipped data ----------
sssv_flipped_back_reversed_back = sssv_flipped_back.copy()
# flip back answers that where reverse coded
mask = (sssv_flipped_back_reversed_back["reverse_coded"] == True)
sssv_flipped_back_reversed_back.loc[mask, "model_answer"] = 3 - sssv_flipped_back_reversed_back.loc[mask, "model_answer"]
sssv_flipped_back_reversed_back.loc[mask, "logit_score"] = 3 - sssv_flipped_back_reversed_back.loc[mask, "logit_score"]
sssv_flipped_back_reversed_back.loc[mask, "prob_score"] = 3 - sssv_flipped_back_reversed_back.loc[mask, "prob_score"]
# drop reverse-coded column (not needed in final data)
#sssv_flipped_back_reversed_back = sssv_flipped_back_reversed_back.drop(columns=["reverse_coded"])



In [ ]:
# organize outcoming dfs

# all reflipped and re-reversed together
SSSV_all_data_reflipped_and_rereversed = pd.concat([sssv_reversed, sssv_flipped_back_reversed_back], ignore_index=True)
all_data_reflipped_and_rereversed = pd.concat([all_data_reflipped_and_rereversed, SSSV_all_data_reflipped_and_rereversed], ignore_index=True)
# all non-reflipped and non-re-reversed together
all_data_raw = pd.concat([all_data_raw, SSSV_data], ignore_index=True)

# only non-flipped data (once re-reversed, once raw) 
no_flip_data_rereversed = pd.concat([no_flip_data_rereversed, sssv_reversed], ignore_index=True)
no_flip_data_raw = pd.concat([no_flip_data_raw, raw_sssv], ignore_index=True)

# only flipped data (once re-reversed, once raw) 
flip_data_rereversed = pd.concat([flip_data_rereversed, sssv_flipped_back_reversed_back], ignore_index=True)
flip_data_raw = pd.concat([flip_data_raw, raw_sssv_flipped], ignore_index=True)

## LOT  TASK
- special: in the tasks, options were strings, not integers! Difficulty for reversing! And need to align with suitable risk measure interpretation!
- therefore direct translation from s1, s2 to 0/1, in which 0 is the non-risky choice and 1 is the risky choice.

In [ ]:
# load data
LOT_data = load_dataframes(task_name="lot", path = PATH)
LOT_data["experiment"] = "LOT"

# rename model_answer_option to model_answer
LOT_data = LOT_data.rename(columns={"model_answer_option": "model_answer"})

In [ ]:
# add argmax logit and probability coloumn
LOT_data = add_argmax_column(LOT_data)

In [ ]:
# for non-numeric answer scales, translate into 0,1 answer alternatives:
# map: s1=0, s2=1, 
cols = ["model_answer", "logit_score", "prob_score"]
LOT_data[cols] = (
    LOT_data[cols]
    .replace({"s1": 1, "s2": 0}) # in lot risky = 
)


In [ ]:
# Which option is the RISKY one. Convention: True => risky is s2, so the raw
# s1->1 mapping gets flipped (1 - x); False => risky is s1 (no flip).
# Ground truth from Frey table S8 + lotteries.csv choices (R vs Decision_X over
# all participants): risky = Option B (s2) for items 1-15, Option A (s1) for
# items 16-25. 
# Regression check: src/preprocessing/check_lot_risky_coding.py
risky_decision_table = {
    1: True,  2: True,  3: True,  4: True,  5: True,  6: True,  7: True,  8: True,  9: True,  10: True,
    11: True, 12: True, 13: True, 14: True, 15: True, 16: False, 17: False, 18: False, 19: False, 20: False,
    21: False, 22: False, 23: False, 24: False, 25: False
}
LOT_data["reverse_coded"] = LOT_data["item_id"].map(risky_decision_table)


In [ ]:
# separate the datasets
raw_lot = LOT_data[LOT_data["flipped"] == False]
raw_lot_flipped = LOT_data[LOT_data["flipped"] == True]

# copy flipped part
lot_flipped_back = raw_lot_flipped.copy()

# flip back
cols = ["model_answer", "logit_score", "prob_score"]
lot_flipped_back[cols] = (
    lot_flipped_back[cols]
    .replace({0: 1, 1: 0})
)

In [ ]:
# reverse human answers (again) where the items where reversed phrased

# ----- for normal data ----------
lot_reversed = raw_lot.copy()
# flip back answers that where reverse coded
mask = (lot_reversed["reverse_coded"] == True)
lot_reversed.loc[mask, "model_answer"] = 1 - lot_reversed.loc[mask, "model_answer"]
lot_reversed.loc[mask, "logit_score"] = 1 - lot_reversed.loc[mask, "logit_score"]
lot_reversed.loc[mask, "prob_score"] = 1 - lot_reversed.loc[mask, "prob_score"]
# drop reverse-coded column (not needed in final data)
#lot_reversed = lot_reversed.drop(columns=["reverse_coded"])


# ----- for re-flipped data ----------
lot_flipped_back_reversed_back = lot_flipped_back.copy()
# flip back answers that where reverse coded
mask = (lot_flipped_back_reversed_back["reverse_coded"] == True)
lot_flipped_back_reversed_back.loc[mask, "model_answer"] = 1 - lot_flipped_back_reversed_back.loc[mask, "model_answer"]
lot_flipped_back_reversed_back.loc[mask, "logit_score"] = 1 - lot_flipped_back_reversed_back.loc[mask, "logit_score"]
lot_flipped_back_reversed_back.loc[mask, "prob_score"] = 1 - lot_flipped_back_reversed_back.loc[mask, "prob_score"]
# drop reverse-coded column (not needed in final data)
#lot_flipped_back_reversed_back = lot_flipped_back_reversed_back.drop(columns=["reverse_coded"])



In [ ]:
# organize outcoming dfs

# all reflipped and re-reversed together
LOT_all_data_reflipped_and_rereversed = pd.concat([lot_reversed, lot_flipped_back_reversed_back], ignore_index=True)
all_data_reflipped_and_rereversed = pd.concat([all_data_reflipped_and_rereversed, LOT_all_data_reflipped_and_rereversed], ignore_index=True)
# all non-reflipped and non-re-reversed together
all_data_raw = pd.concat([all_data_raw, LOT_data], ignore_index=True)

# only non-flipped data (once re-reversed, once raw) 
no_flip_data_rereversed = pd.concat([no_flip_data_rereversed, lot_reversed], ignore_index=True)
no_flip_data_raw = pd.concat([no_flip_data_raw, raw_lot], ignore_index=True)

# only flipped data (once re-reversed, once raw) 
flip_data_rereversed = pd.concat([flip_data_rereversed, lot_flipped_back_reversed_back], ignore_index=True)
flip_data_raw = pd.concat([flip_data_raw, raw_lot_flipped], ignore_index=True)

## DFD TASK

In [ ]:
# load data
DFD_data = load_dataframes(task_name="dfd", path = PATH)
DFD_data["experiment"] = "DFD"

# rename model_answer_option to model_answer
DFD_data = DFD_data.rename(columns={"model_answer_option": "model_answer"})
DFD_data = DFD_data.rename(columns={"gamble_id": "item_id"})

In [ ]:
# add argmax logit and probability coloumn
DFD_data = add_argmax_column(DFD_data)

In [ ]:
# for non-numeric answer scales, translate into 0,1 answer alternatives:
# map: s1=01 s2=0, 1 means risky choice, 0 not

# flip back
cols = ["model_answer", "logit_score", "prob_score"]
DFD_data[cols] = (
    DFD_data[cols]
    .replace({"s1": 1, "s2": 0}) # in dfd usually the first option is the risky one, except when reversed or flipped
)


In [ ]:
# add which item was reverse coded
risky_decision_table = {
    "he04_1": False,
    "he04_2": False,
    "he04_3": True,
    "he04_4": True,
    "he04_5": False,
    "he04_6": False,
    "he04_2inv": True,
    "he04_6inv": True
}

DFD_data["reverse_coded"] = DFD_data["item_id"].map(risky_decision_table)


In [ ]:
# seperate the datasets
raw_dfd = DFD_data[DFD_data["flipped"] == False]
raw_dfd_flipped = DFD_data[DFD_data["flipped"] == True]
dfd_flipped_back = raw_dfd_flipped.copy()


# flip back
cols = ["model_answer", "logit_score", "prob_score"]
dfd_flipped_back[cols] = (
    dfd_flipped_back[cols]
    .replace({0: 1, 1: 0})
)

In [ ]:
# reverse human answers (again) where the items where reversed phrased


# ----- for normal data ----------
dfd_reversed = raw_dfd.copy()
# flip back answers that where reverse coded
mask = (dfd_reversed["reverse_coded"] == True)
dfd_reversed.loc[mask, "model_answer"] = 1 - dfd_reversed.loc[mask, "model_answer"]
dfd_reversed.loc[mask, "logit_score"] = 1 - dfd_reversed.loc[mask, "logit_score"]
dfd_reversed.loc[mask, "prob_score"] = 1 - dfd_reversed.loc[mask, "prob_score"]
# drop reverse-coded column (not needed in final data)
#dfd_reversed = dfd_reversed.drop(columns=["reverse_coded"])


# ----- for re-flipped data ----------
dfd_flipped_back_reversed_back = dfd_flipped_back.copy()
# flip back answers that where reverse coded
mask = (dfd_flipped_back_reversed_back["reverse_coded"] == True)
dfd_flipped_back_reversed_back.loc[mask, "model_answer"] = 1 - dfd_flipped_back_reversed_back.loc[mask, "model_answer"]
dfd_flipped_back_reversed_back.loc[mask, "logit_score"] = 1 - dfd_flipped_back_reversed_back.loc[mask, "logit_score"]
dfd_flipped_back_reversed_back.loc[mask, "prob_score"] = 1 - dfd_flipped_back_reversed_back.loc[mask, "prob_score"]
# drop reverse-coded column (not needed in final data)
#dfd_flipped_back_reversed_back = dfd_flipped_back_reversed_back.drop(columns=["reverse_coded"])



In [ ]:
# organize outcoming dfs

# all reflipped and re-reversed together
DFD_all_data_reflipped_and_rereversed = pd.concat([dfd_reversed, dfd_flipped_back_reversed_back], ignore_index=True)
all_data_reflipped_and_rereversed = pd.concat([all_data_reflipped_and_rereversed, DFD_all_data_reflipped_and_rereversed], ignore_index=True)
# all non-reflipped and non-re-reversed together
all_data_raw = pd.concat([all_data_raw, DFD_data], ignore_index=True)

# only non-flipped data (once re-reversed, once raw) 
no_flip_data_rereversed = pd.concat([no_flip_data_rereversed, dfd_reversed], ignore_index=True)
no_flip_data_raw = pd.concat([no_flip_data_raw, raw_dfd], ignore_index=True)

# only flipped data (once re-reversed, once raw) 
flip_data_rereversed = pd.concat([flip_data_rereversed, dfd_flipped_back_reversed_back], ignore_index=True)
flip_data_raw = pd.concat([flip_data_raw, raw_dfd_flipped], ignore_index=True)

## MPL TASK


In [ ]:
# load data
MPL_data = load_dataframes(task_name="mpl", path = PATH)
MPL_data["experiment"] = "MPL"

# rename model_answer_option to model_answer
MPL_data = MPL_data.rename(columns={"model_answer_option": "model_answer"})
MPL_data = MPL_data.rename(columns={"gamble_id": "item_id"})

In [ ]:
# add argmax logit and probability coloumn
MPL_data = add_argmax_column(MPL_data)

In [ ]:
# for non-numeric answer scales, translate into 0,1 answer alternatives:
# map: s1=0, s2=1, 1 means risky choice, 0 not

# flip back
cols = ["model_answer", "logit_score", "prob_score"]
MPL_data[cols] = (
    MPL_data[cols]
    .replace({"s1": 0, "s2": 1}) # in mple, always the second option (unflipped) is the risky one
)


In [ ]:
# seperate the datasets
raw_mpl = MPL_data[MPL_data["flipped"] == False]
raw_mpl_flipped = MPL_data[MPL_data["flipped"] == True]
mpl_flipped_back = raw_mpl_flipped.copy()


# flip back
cols = ["model_answer", "logit_score", "prob_score"]
mpl_flipped_back[cols] = (
    mpl_flipped_back[cols]
    .replace({0: 1, 1: 0})
)

In [ ]:
# organize outcoming dfs

# all reflipped and re-reversed together
MPL_all_data_reflipped = pd.concat([raw_mpl, mpl_flipped_back], ignore_index=True)
all_data_reflipped_and_rereversed = pd.concat([all_data_reflipped_and_rereversed, MPL_all_data_reflipped], ignore_index=True)
# all non-reflipped and non-re-reversed together
all_data_raw = pd.concat([all_data_raw, MPL_data], ignore_index=True)

# only non-flipped data (once re-reversed, once raw) 
no_flip_data_rereversed = pd.concat([no_flip_data_rereversed, raw_mpl], ignore_index=True)
no_flip_data_raw = pd.concat([no_flip_data_raw, raw_mpl], ignore_index=True)

# only flipped data (once re-reversed, once raw) 
flip_data_rereversed = pd.concat([flip_data_rereversed, mpl_flipped_back], ignore_index=True)
flip_data_raw = pd.concat([flip_data_raw, raw_mpl_flipped], ignore_index=True)

# Saving new processed dataframe

In [ ]:
# save data
all_data_reflipped_and_rereversed.to_csv('../../data/intermediate/risk_data/LLM_data_proc_prompts_direct/LLM_all_data_reflipped_and_rereversed.csv', index=False)
all_data_raw.to_csv('../../data/intermediate/risk_data/LLM_data_proc_prompts_direct/LLM_all_data_raw.csv', index=False)

no_flip_data_rereversed.to_csv('../../data/intermediate/risk_data/LLM_data_proc_prompts_direct/LLM_no_flip_data_rereversed.csv', index=False)
no_flip_data_raw.to_csv('../../data/intermediate/risk_data/LLM_data_proc_prompts_direct/LLM_no_flip_data_raw.csv', index=False)

flip_data_rereversed.to_csv('../../data/intermediate/risk_data/LLM_data_proc_prompts_direct/LLM_flip_data_rereversed.csv', index=False)
flip_data_raw.to_csv('../../data/intermediate/risk_data/LLM_data_proc_prompts_direct/LLM_flip_data_raw.csv', index=False)
